# is_title 분류기 — KoELECTRA 이진 분류 파인튜닝

**목표**: 가정통신문 문장을 받아 `제목(1)` vs `제목 아님(0)` 으로 이진 분류

| 항목 | 내용 |
|------|------|
| 모델 | `monologg/koelectra-small-v3-discriminator` |
| 데이터 | `v3_dual_labeled_clean.jsonl` — `{text, is_todo, is_title}` 포맷 |
| 출력 클래스 | 0: 제목 아님, 1: 제목 |
| 저장 위치 | `checkpoints/koelectra-title/` |

> **주의**: 양성(제목) 비율이 약 1.8% (519/28,890) 로 매우 불균형.
> `compute_class_weight('balanced')` 로 자동 보정하며, 추론 임계값은 0.4 사용.

**실행 방법**
1. `런타임` → `런타임 유형 변경` → **T4 GPU** 선택
2. `v3_dual_labeled_clean.jsonl` 파일을 Colab에 업로드
3. 셀을 위에서부터 순서대로 실행
4. 마지막 셀에서 `koelectra-title.zip` 다운로드 → `model/extraction/checkpoints/koelectra-title/` 에 압축 풀기

## 1. 라이브러리 설치

In [ ]:
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.0 evaluate==0.4.3 scikit-learn

## 2. GPU 확인

In [ ]:
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 이름:', torch.cuda.get_device_name(0))

## 3. 데이터 로드

`v3_dual_labeled_clean.jsonl` 포맷: `{"text": str, "is_todo": bool, "is_title": bool}`

- `is_title` 필드만 라벨로 사용 (`is_todo` 는 무시)
- 10자 미만 / 5000자 초과는 clean 파일에서 이미 제거됨

In [ ]:
import json
from collections import Counter

DATA_PATH = 'v3_dual_labeled_clean.jsonl'

texts: list[str] = []
labels: list[int] = []

with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = str(obj.get('text', '') or '').strip()
        if not text:
            continue
        texts.append(text)
        labels.append(int(bool(obj.get('is_title', False))))

cnt = Counter(labels)
print(f'총 문장 수  : {len(texts)}')
print(f'라벨 분포:')
print(f'  0 (제목 아님) : {cnt[0]}  ({cnt[0]/len(labels)*100:.1f}%)')
print(f'  1 (제목)      : {cnt[1]}  ({cnt[1]/len(labels)*100:.1f}%)')

## 4. Train/Val 분할 (stratified)

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

print(f'학습셋: {len(train_texts)}')
print(f'  제목: {sum(train_labels)}, 제목 아님: {len(train_labels) - sum(train_labels)}')
print(f'검증셋: {len(val_texts)}')
print(f'  제목: {sum(val_labels)}, 제목 아님: {len(val_labels) - sum(val_labels)}')

## 5. 토크나이저 + 데이터셋

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'monologg/koelectra-small-v3-discriminator'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = (
    Dataset.from_dict({'text': train_texts, 'label': train_labels})
    .map(encode, batched=True)
    .remove_columns(['text'])
)
val_ds = (
    Dataset.from_dict({'text': val_texts, 'label': val_labels})
    .map(encode, batched=True)
    .remove_columns(['text'])
)

print('학습 데이터셋:', train_ds)
print('검증 데이터셋:', val_ds)

## 6. 모델 + 클래스 가중치

양성(제목) 비율 ~1.8% → `compute_class_weight('balanced')` 로 자동 보정.
제목 클래스 가중치가 약 27~30배 높게 설정됩니다.

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

id2label = {0: '제목 아님', 1: '제목'}
label2id = {'제목 아님': 0, '제목': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

_w = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
_class_weights = torch.tensor(_w, dtype=torch.float)
print(f'클래스 가중치: 제목 아님={_w[0]:.3f}, 제목={_w[1]:.3f}')


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fn = torch.nn.CrossEntropyLoss(weight=_class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1':        f1_score(labels, preds, pos_label=1, zero_division=0),
        'precision': precision_score(labels, preds, pos_label=1, zero_division=0),
        'recall':    recall_score(labels, preds, pos_label=1, zero_division=0),
    }

## 7. 학습 실행

에폭 15회 (is_todo 대비 +5 — 양성 샘플이 적어 더 오래 학습).
T4 GPU 기준 약 10~15분.

In [ ]:
from transformers import TrainingArguments, DataCollatorWithPadding

args = TrainingArguments(
    output_dir='./koelectra-title-output',
    save_safetensors=False,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 8. 최종 평가

In [ ]:
from sklearn.metrics import classification_report

preds_out = trainer.predict(val_ds)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids

print(classification_report(
    y_true, y_pred,
    target_names=['제목 아님', '제목'],
    digits=4,
    zero_division=0,
))

## 9. 임계값 분석

양성 비율이 낮으므로 기본 0.5 대신 최적 임계값을 찾습니다.
**Recall이 낮으면 임계값을 낮추세요** (0.3~0.4 권장).

In [ ]:
import torch.nn.functional as F

probs = torch.softmax(torch.tensor(preds_out.predictions), dim=-1)[:, 1].numpy()

print(f'{'임계값':>6}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}')
print('-' * 40)
for thresh in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (probs >= thresh).astype(int)
    pr = precision_score(y_true, p, pos_label=1, zero_division=0)
    rc = recall_score(y_true, p, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, p, pos_label=1, zero_division=0)
    print(f'{thresh:>6.1f}  {pr:>10.4f}  {rc:>8.4f}  {f1:>8.4f}')

## 10. 모델 저장

In [ ]:
OUTPUT_DIR = './koelectra-title'
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('저장 완료:', OUTPUT_DIR)
!ls -lh {OUTPUT_DIR}

## 11. 추론 테스트

`predict.py` 의 `extract_title()` 은 체크포인트가 있으면 ML 모델을 자동 사용합니다.
임계값은 위 분석 결과를 반영해 조정하세요 (기본값 0.4).

In [ ]:
from transformers import AutoModelForSequenceClassification as AM

TITLE_THRESHOLD = 0.4
_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
_clf = AM.from_pretrained(OUTPUT_DIR, num_labels=2)
_clf.eval()

def title_predict(sentence: str) -> tuple[int, float]:
    inputs = _tok(sentence, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        prob = torch.softmax(_clf(**inputs).logits, dim=-1)[0]
    label = 1 if prob[1].item() >= TITLE_THRESHOLD else 0
    return label, round(float(prob[1].item()), 3)

test_cases = [
    ('2026 해원 놀이 한마당 안내',             True,  '명확한 제목'),
    ('겨울방학 돌봄교실 중식비 수납 안내 제2021-264호', True,  '공문번호 포함 제목'),
    ('학부모님 안녕하십니까?',                 False, '인사말'),
    ('4월 30일까지 동의서를 제출해주세요.',    False, 'TODO 문장'),
    ('준비물: 실내화, 도시락',                 False, '준비물 항목'),
    ('1. 행사 개요',                           False, '항목 번호'),
]

print(f'{'문장':<44} {'예상':>6} {'예측':>6} {'P(제목)':>8}')
print('-' * 68)
for sent, expected, desc in test_cases:
    lbl, prob = title_predict(sent)
    ok = (lbl == 1) == expected
    mark = '✅' if ok else '❌'
    tag = '제목' if lbl == 1 else '아님'
    print(f'{mark} {sent:<42} {str(expected):>6} {tag:>6} {prob:>8.3f}  ({desc})')

## 12. 압축 + 다운로드

In [ ]:
!zip -r koelectra-title.zip koelectra-title/

from google.colab import files
files.download('koelectra-title.zip')

## 끝

`koelectra-title.zip` → 압축 풀기 → `model/extraction/checkpoints/koelectra-title/`

`predict.py` 의 `extract_title()` 이 체크포인트를 자동 감지해서 ML 모델로 전환합니다.
체크포인트가 없으면 heuristic(`is_title_heuristic`)으로 fallback됩니다.

### predict.py에서 임계값 수정 위치

```python
# model/extraction/file/predict.py
TITLE_THRESHOLD = 0.4   # 위 임계값 분석 결과에 따라 조정
```